In [1]:
import duckdb

con = duckdb.connect()

PROCESSED = "C:/Users/HP/Downloads/DS/da9/walmart-demand-forecast/data/processed"

con.execute(f"""
    CREATE OR REPLACE VIEW walmart AS
    SELECT * FROM read_parquet('{PROCESSED}/walmart_joined.parquet')
""")

# --- View đã ép kiểu số cho các cột bị đọc thành VARCHAR ---
con.execute("""
    CREATE OR REPLACE VIEW walmart_typed AS
    SELECT
        Store, Dept, Date, Weekly_Sales, IsHoliday,
        TRY_CAST(Temperature AS DOUBLE) AS Temperature,
        TRY_CAST(Fuel_Price AS DOUBLE) AS Fuel_Price,
        TRY_CAST(MarkDown1 AS DOUBLE) AS MarkDown1,
        TRY_CAST(MarkDown2 AS DOUBLE) AS MarkDown2,
        TRY_CAST(MarkDown3 AS DOUBLE) AS MarkDown3,
        TRY_CAST(MarkDown4 AS DOUBLE) AS MarkDown4,
        TRY_CAST(MarkDown5 AS DOUBLE) AS MarkDown5,
        TRY_CAST(CPI AS DOUBLE) AS CPI,
        TRY_CAST(Unemployment AS DOUBLE) AS Unemployment,
        Store_Type, Store_Size
    FROM walmart
""")

# --- Investigate 1: phân phối thực tế của MarkDown1 ---
md_dist = con.execute("""
    SELECT
        MIN(MarkDown1) AS min1,
        MAX(MarkDown1) AS max1,
        AVG(MarkDown1) AS mean1,
        SUM(CASE WHEN MarkDown1 = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_zero1,
        MIN(Date) FILTER (WHERE MarkDown1 > 0) AS first_nonzero_md1_date
    FROM walmart_typed
""").df()
print("=== MARKDOWN1 DISTRIBUTION ===")
print(md_dist)

# --- Investigate 2: MarkDown1 trước/sau mốc 2011-11 ---
md_before_after = con.execute("""
    SELECT
        CASE WHEN Date < DATE '2011-11-01' THEN 'before_2011_11' ELSE 'after_2011_11' END AS period,
        COUNT(*) AS n_rows,
        SUM(CASE WHEN MarkDown1 IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_null_md1,
        AVG(MarkDown1) AS avg_md1_when_present
    FROM walmart_typed
    GROUP BY period
""").df()
print("\n=== MARKDOWN1 NULL: TRƯỚC VS SAU 2011-11 ===")
print(md_before_after)

# --- Investigate 3: phân phối số tuần theo Store-Dept ---
weeks_dist = con.execute("""
    WITH weeks_per_pair AS (
        SELECT Store, Dept, COUNT(DISTINCT Date) AS n_weeks
        FROM walmart_typed
        GROUP BY Store, Dept
    )
    SELECT
        CASE
            WHEN n_weeks < 10 THEN '<10 tuần (loại bỏ)'
            WHEN n_weeks < 52 THEN '10-51 tuần (chưa đủ 1 năm)'
            WHEN n_weeks < 104 THEN '52-103 tuần (chưa đủ 2 chu kỳ mùa vụ)'
            ELSE '>=104 tuần (đủ để SARIMA)'
        END AS bucket,
        COUNT(*) AS n_store_dept_pairs
    FROM weeks_per_pair
    GROUP BY bucket
    ORDER BY n_store_dept_pairs DESC
""").df()
print("\n=== PHÂN PHỐI SỐ TUẦN THEO STORE-DEPT ===")
print(weeks_dist)

# --- Investigate 4: Weekly_Sales âm ---
negative_sales = con.execute("""
    SELECT
        COUNT(*) AS n_negative,
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM walmart_typed) AS pct_negative,
        MIN(Weekly_Sales) AS most_negative_value
    FROM walmart_typed
    WHERE Weekly_Sales < 0
""").df()
print("\n=== WEEKLY_SALES ÂM ===")
print(negative_sales)


con.execute(f"""
    COPY walmart_typed TO '{PROCESSED}/walmart_clean.parquet' (FORMAT PARQUET)
""")
print("\n=== ĐÃ LƯU walmart_clean.parquet ===")

=== MARKDOWN1 DISTRIBUTION ===
   min1      max1        mean1  pct_zero1 first_nonzero_md1_date
0  0.27  88646.76  7246.420196        0.0             2011-11-11

=== MARKDOWN1 NULL: TRƯỚC VS SAU 2011-11 ===
           period  n_rows  pct_null_md1  avg_md1_when_present
0  before_2011_11  267184    100.000000                   NaN
1   after_2011_11  154386      2.399829           7246.420196

=== PHÂN PHỐI SỐ TUẦN THEO STORE-DEPT ===
                                  bucket  n_store_dept_pairs
0              >=104 tuần (đủ để SARIMA)                2868
1             10-51 tuần (chưa đủ 1 năm)                 176
2                     <10 tuần (loại bỏ)                 164
3  52-103 tuần (chưa đủ 2 chu kỳ mùa vụ)                 123

=== WEEKLY_SALES ÂM ===
   n_negative  pct_negative  most_negative_value
0        1285      0.304813             -4988.94

=== ĐÃ LƯU walmart_clean.parquet ===
